# 🎛️ Mai AI – Advanced Container Controller

Welcome to the **Mai AI Container Controller**! This Jupyter Notebook application offers a professional-grade interactive dashboard to monitor, manage, and create Docker containers directly from your notebook cells.

### ✨ Key Features:
1. **Tabbed Dashboard**: Separate panels for Management, Container Creation, Real-time Logging, and System Resources.
2. **Resilience & Safety**: Connection guards catch Docker connection errors gracefully, displaying a retry option instead of crashing.
3. **Real-time Monitoring**: Multi-threaded updates with customizable refresh rates.
4. **Smart Keyboard Fix**: Injected JavaScript prevents keyboard inputs in fields from triggering unwanted notebook shortcuts.

---

In [3]:
import docker
import os
import sys
import time
import threading
import multiprocessing
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==============================================================================
# 0. GLOBAL CONFIGURATION & FALLBACKS
# ==============================================================================
CONFIG = {
    "DEFAULT_IMAGE": "python:3.11-slim",
    "CONTAINER_WORK_DIR": "/app",
    "RESOURCES": {
        "MIN_CPUS": 1,
        "KEEP_FREE_CPU_PERCENT": 20
    }
}

# Try to discover project root folder index if present
try:
    from folder_index import FOLDER_STRUCTURE, PROJECT_ROOT
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.getcwd())
    FOLDER_STRUCTURE = {"root": PROJECT_ROOT}

# State variables
docker_client = None
live_monitor_thread = None
live_monitor_active = False

# ==============================================================================
# 1. CUSTOM STYLING (PREMIUM UI BRANDING)
# ==============================================================================
UI_STYLE = """
<style>
    .controller-title { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #1e293b; margin-bottom: 5px; }
    .status-badge { padding: 4px 8px; border-radius: 4px; font-weight: bold; font-family: monospace; font-size: 11px; }
    .status-running { background-color: #dcfce7; color: #166534; border: 1px solid #bbf7d0; }
    .status-stopped { background-color: #fee2e2; color: #991b1b; border: 1px solid #fecaca; }
    .status-paused { background-color: #fef9c3; color: #854d0e; border: 1px solid #fef08a; }
    .console-box { font-family: 'Consolas', monospace; font-size: 12px; background-color: #0f172a; color: #38bdf8; padding: 12px; border-radius: 6px; border: 1px solid #1e293b; height: 180px; overflow-y: auto; }
    .warning-card { background-color: #fffbeb; border-left: 5px solid #d97706; padding: 15px; border-radius: 4px; font-family: system-ui; margin-bottom: 10px; }
    .success-card { background-color: #f0fdf4; border-left: 5px solid #16a34a; padding: 15px; border-radius: 4px; font-family: system-ui; margin-bottom: 10px; }
</style>
"""
display(HTML(UI_STYLE))

# ==============================================================================
# 2. DOCKER CONNECTION GUARD
# ==============================================================================
def connect_docker():
    global docker_client
    try:
        docker_client = docker.from_env()
        docker_client.ping()
        return True
    except Exception as e:
        docker_client = None
        return False

# Initial connection check
is_connected = connect_docker()


In [4]:
# ==============================================================================
# 3. UI INITIALIZATION & LAYOUT BUILD
# ==============================================================================

# Global Outputs
ui_output_console = widgets.Output()
ui_monitor_panel = widgets.Output(layout=widgets.Layout(border='1px solid #cbd5e1', padding='10px', margin='10px 0', border_radius='6px', background_color='#f8fafc'))

# Custom logger helper
def log_to_console(message, type="info"):
    timestamp = time.strftime("%H:%M:%S")
    color_map = {
        "info": "#38bdf8",
        "success": "#4ade80",
        "warning": "#facc15",
        "danger": "#f87171"
    }
    prefix_map = {"info": "ℹ️ [INFO]", "success": "✅ [OK]", "warning": "⚠️ [WARN]", "danger": "❌ [FAIL]"}
    
    prefix = prefix_map.get(type, "")
    color = color_map.get(type, "#f8fafc")
    formatted_line = f"<div style='color: {color}; margin-bottom: 2px;'>[{timestamp}] {prefix} {message}</div>"
    
    with ui_output_console:
        # Rather than completely clearing, we show structured entries
        display(HTML(formatted_line))

def clear_console(b=None):
    with ui_output_console:
        clear_output()
    log_to_console("Console cleared.")

# --- 1. Tab widgets & Dropdowns ---
dropdown_containers = widgets.Dropdown(description='📦 Select Target:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%'))

def populate_containers():
    if not is_connected:
        dropdown_containers.options = [("⚠️ Docker Daemon offline", "none")]
        return
    try:
        containers = docker_client.containers.list(all=True)
        if not containers:
            dropdown_containers.options = [("No containers found", "none")]
        else:
            dropdown_containers.options = [(f"{c.name} ({c.status.upper()})", c.name) for c in containers]
    except Exception as e:
        dropdown_containers.options = [("⚠️ Connection lost", "none")]
        log_to_console(f"Failed to fetch containers: {e}", "danger")

# --- TAB 1: MANAGEMENT & POWER CONTROLS ---
btn_power_start = widgets.Button(description="▶️ Start", button_style="success", layout=widgets.Layout(flex='1 1 auto', height='36px', min_width='80px'))
btn_power_stop = widgets.Button(description="🛑 Stop", button_style="danger", layout=widgets.Layout(flex='1 1 auto', height='36px', min_width='80px'))
btn_power_pause = widgets.Button(description="⏸️ Pause", button_style="warning", layout=widgets.Layout(flex='1 1 auto', height='36px', min_width='80px'))
btn_power_unpause = widgets.Button(description="⏯️ Unpause", button_style="info", layout=widgets.Layout(flex='1 1 auto', height='36px', min_width='80px'))
btn_power_restart = widgets.Button(description="🔄 Restart", layout=widgets.Layout(flex='1 1 auto', height='36px', min_width='80px'))
btn_power_remove = widgets.Button(description="🗑️ Delete", button_style="danger", layout=widgets.Layout(flex='1 1 auto', height='36px', min_width='80px'))
btn_refresh_list = widgets.Button(description="🔄 Reload List", button_style="primary", layout=widgets.Layout(width='120px'))

# Live monitor controls
toggle_monitor = widgets.ToggleButton(value=False, description="📊 Enable Live Monitor", button_style="info", icon='heartbeat', layout=widgets.Layout(width='200px'))
slider_refresh = widgets.IntSlider(value=2, min=1, max=10, description='Rate (Sec):', layout=widgets.Layout(width='250px'))

# Layout Tab 1
controls_box = widgets.HBox([btn_power_start, btn_power_stop, btn_power_pause, btn_power_unpause, btn_power_restart, btn_power_remove], layout=widgets.Layout(justify_content='space-between', margin='10px 0'))
monitor_controls_box = widgets.HBox([toggle_monitor, slider_refresh], layout=widgets.Layout(align_items='center', margin='5px 0'))
tab_management = widgets.VBox([
    widgets.HTML("<h4>🎚️ Active Container Controls</h4>"),
    widgets.HBox([dropdown_containers, btn_refresh_list], layout=widgets.Layout(align_items='center', margin='5px 0')),
    controls_box,
    widgets.HTML("<hr style='border-color: #e2e8f0; margin: 10px 0;'/>"),
    widgets.HTML("<h4>📈 Resource & Status Stream</h4>"),
    monitor_controls_box,
    ui_monitor_panel
])

# --- TAB 2: PRO-GRADE CONTAINER CREATOR ---
input_create_name = widgets.Text(value='mai_ai_workspace', description='Name:', placeholder='container_name')
input_create_image = widgets.Text(value=CONFIG["DEFAULT_IMAGE"], description='Docker Image:', placeholder='python:3.11-slim')
input_host_port = widgets.IntText(value=8000, description='Host Port:')
input_container_port = widgets.IntText(value=8000, description='Target Port:')

# Resource limiters
cpu_limit = widgets.FloatSlider(value=1.0, min=0.5, max=float(multiprocessing.cpu_count()), step=0.5, description='CPU Limit:', style={'description_width': 'initial'})
ram_limit = widgets.SelectionSlider(
    options=[('128 MB', '128m'), ('256 MB', '256m'), ('512 MB', '512m'), ('1 GB', '1024m'), ('2 GB', '2048m'), ('4 GB', '4096m'), ('8 GB', '8192m')],
    value='512m',
    description='RAM Limit:',
    style={'description_width': 'initial'}
)

# Volume mount options
input_host_volume = widgets.Text(value=FOLDER_STRUCTURE["root"], description='Host Dir (Bind):', style={'description_width': 'initial'})
input_target_volume = widgets.Text(value=CONFIG["CONTAINER_WORK_DIR"], description='Container Dir:', style={'description_width': 'initial'})
checkbox_volume = widgets.Checkbox(value=True, description='Mount project workspace volume', style={'description_width': 'initial'})
checkbox_tty = widgets.Checkbox(value=True, description='Allocate Pseudo-TTY (Interactive / keep alive)', style={'description_width': 'initial'})

btn_launch_container = widgets.Button(description="🚀 Launch New Container", button_style="success", layout=widgets.Layout(height='40px', font_weight='bold'))

# Form design layout
form_column_left = widgets.VBox([input_create_name, input_create_image, input_host_port, input_container_port], layout=widgets.Layout(flex='1 1 50%', padding='10px'))
form_column_right = widgets.VBox([cpu_limit, ram_limit, input_host_volume, input_target_volume, checkbox_volume, checkbox_tty], layout=widgets.Layout(flex='1 1 50%', padding='10px'))
tab_creator = widgets.VBox([
    widgets.HTML("<h4>🛠️ Configure New Container Deployment</h4>"),
    widgets.HBox([form_column_left, form_column_right]),
    widgets.HTML("<hr style='border-color: #e2e8f0; margin: 10px 0;'/>"),
    btn_launch_container
])

# --- TAB 3: LIVE LOG MONITOR ---
log_num_lines = widgets.IntSlider(value=20, min=5, max=100, step=5, description='Tail Lines:', layout=widgets.Layout(width='300px'))
log_search_filter = widgets.Text(value='', description='Filter Logs:', placeholder='e.g., Error, Connection', layout=widgets.Layout(width='300px'))
btn_refresh_logs = widgets.Button(description="📋 Refresh Logs", button_style="info", layout=widgets.Layout(width='150px'))
ui_logs_area = widgets.Output(layout=widgets.Layout(border='1px solid #1e293b', background_color='#0f172a', padding='10px', height='300px', overflow_y='auto', border_radius='4px'))

tab_logs = widgets.VBox([
    widgets.HTML("<h4>📋 Container Log Streams</h4>"),
    widgets.HBox([log_num_lines, log_search_filter, btn_refresh_logs], layout=widgets.Layout(align_items='center', justify_content='space-between', margin='5px 0')),
    widgets.HTML("<div style='margin-bottom: 5px; font-size:11px; color:#64748b;'>Scrollable container live-output dashboard:</div>"),
    ui_logs_area
])

# --- TAB 4: SYSTEM RESOURCES OVERVIEW ---
ui_system_area = widgets.Output()
btn_refresh_system = widgets.Button(description="🔌 Query System Stats", button_style="primary", layout=widgets.Layout(width='200px'))

tab_system = widgets.VBox([
    widgets.HTML("<h4>⚙️ Docker Host Environment Overview</h4>"),
    btn_refresh_system,
    widgets.HTML("<hr style='border-color: #e2e8f0; margin: 10px 0;'/>"),
    ui_system_area
])

# --- COMBINE INTO MAIN TABS INTERFACE ---
main_tabs = widgets.Tab()
main_tabs.children = [tab_management, tab_creator, tab_logs, tab_system]
main_tabs.set_title(0, '📦 Container Manager')
main_tabs.set_title(1, '🛠️ Creator Studio')
main_tabs.set_title(2, '📋 Log Streams')
main_tabs.set_title(3, '🖥️ System Overview')

btn_clear_console = widgets.Button(description="🧹 Clear Console Logs", layout=widgets.Layout(width='180px', margin='5px 0'))
btn_clear_console.on_click(clear_console)

ui_footer_and_console = widgets.VBox([
    widgets.HTML("<h4>📟 Action Output Feed</h4>"),
    ui_output_console,
    btn_clear_console
])

# Container UI Frame wrapper
main_dashboard = widgets.VBox([
    widgets.HTML("<h2 class='controller-title'>🎛️ Mai AI – Pro Container Controller</h2>"),
    main_tabs,
    widgets.HTML("<hr style='border-color: #cbd5e1; margin: 15px 0;'/>"),
    ui_footer_and_console
], layout=widgets.Layout(border='1px solid #94a3b8', padding='15px', max_width='800px', border_radius='8px', background_color='#f8fafc'))


In [6]:
# ==============================================================================
# 4. BUSINESS LOGIC & INTERACTION CONTROLLERS
# ==============================================================================

def handle_container_action(b):
    target = dropdown_containers.value
    if not target or target == "none":
        log_to_console("Please select a valid container first!", "warning")
        return
        
    action = b.description
    log_to_console(f"Executing '{action}' action on container '{target}'...", "info")
    
    try:
        c = docker_client.containers.get(target)
        
        if "Start" in action:
            c.start()
            log_to_console(f"Container '{target}' started successfully.", "success")
        elif "Stop" in action:
            c.stop()
            log_to_console(f"Container '{target}' stopped safely.", "success")
        elif "Pause" in action:
            c.pause()
            log_to_console(f"Container '{target}' paused.", "success")
        elif "Unpause" in action:
            c.unpause()
            log_to_console(f"Container '{target}' unpaused.", "success")
        elif "Restart" in action:
            c.restart()
            log_to_console(f"Container '{target}' restarted cleanly.", "success")
        elif "Delete" in action:
            # Prompt confirmation safely or just force remove
            c.remove(force=True)
            log_to_console(f"Container '{target}' force-removed.", "success")
            
        # Refresh elements after action
        populate_containers()
        refresh_logs_panel()
    except Exception as e:
        log_to_console(f"Action failed: {e}", "danger")

def deploy_new_container(b):
    if not is_connected:
        log_to_console("Docker Daemon is disconnected. Cannot create container.", "danger")
        return
        
    name = input_create_name.value.strip()
    image = input_create_image.value.strip()
    host_p = input_host_port.value
    target_p = input_container_port.value
    
    if not name or not image:
        log_to_console("Container name and image template must be supplied!", "warning")
        return
        
    # Validate resources
    total_cpus = multiprocessing.cpu_count()
    max_recommended = float(total_cpus * (1 - CONFIG["RESOURCES"]["KEEP_FREE_CPU_PERCENT"]/100))
    if cpu_limit.value > max_recommended:
        log_to_console(f"Resource Warning: Restricting CPU limit to recommended max ({max_recommended:.1f} Cores) to prevent host lockup.", "warning")
    
    log_to_console(f"Pulling image '{image}' if not present locally (this may take a minute)...", "info")
    
    try:
        # Prepare ports mapping if specified
        ports = {f"{target_p}/tcp": host_p} if (host_p > 0 and target_p > 0) else None
        
        # Prepare volume binding
        volumes = None
        if checkbox_volume.value:
            h_dir = input_host_volume.value.strip()
            t_dir = input_target_volume.value.strip()
            if h_dir and t_dir:
                volumes = {h_dir: {'bind': t_dir, 'mode': 'rw'}}
                
        # Convert CPU to nano_cpus (Docker SDK expects nano_cpus: 1 CPU Core = 1e9 nano_cpus)
        nano_cpus = int(cpu_limit.value * 1e9)
        
        # Spin container up
        new_c = docker_client.containers.run(
            image=image,
            name=name,
            detach=True,
            tty=checkbox_tty.value,
            ports=ports,
            volumes=volumes,
            nano_cpus=nano_cpus,
            mem_limit=ram_limit.value,
            working_dir=input_target_volume.value if volumes else None
        )
        
        log_to_console(f"Container '{new_c.name}' successfully deployed and online!", "success")
        populate_containers()
        # Select newly created container
        dropdown_containers.value = new_c.name
    except Exception as e:
        log_to_console(f"Deployment aborted: {e}", "danger")

def refresh_logs_panel(b=None):
    target = dropdown_containers.value
    with ui_logs_area:
        clear_output()
        if not target or target == "none":
            print("No container targeted for streaming logs.")
            return
        try:
            c = docker_client.containers.get(target)
            lines = log_num_lines.value
            logs_raw = c.logs(tail=lines).decode("utf-8", errors='replace')
            
            if not logs_raw.strip():
                print("[EMPTY STREAM] Log storage has no lines yet.")
                return
                
            keyword = log_search_filter.value.strip().lower()
            if keyword:
                filtered_lines = [line for line in logs_raw.splitlines() if keyword in line.lower()]
                if filtered_lines:
                    print(f"--- FILTERED LOGS (Keyword: '{keyword}') ---")
                    print("\n".join(filtered_lines))
                else:
                    print(f"--- FILTERED LOGS (Keyword: '{keyword}') ---\n[No matches found]")
            else:
                print(logs_raw)
        except Exception as e:
            print(f"Failed to load log stream: {e}")

def refresh_system_panel(b=None):
    with ui_system_area:
        clear_output()
        if not is_connected:
            display(HTML("<div class='warning-card'>⚠️ Docker Daemon offline. Verify installation is running.</div>"))
            return
        try:
            sys_info = docker_client.info()
            vols = docker_client.volumes.list()
            nets = docker_client.networks.list()
            
            html_stats = f"""
            <table style='width: 100%; border-collapse: collapse; font-family: system-ui;'>
                <tr style='border-bottom: 1px solid #cbd5e1; padding: 6px;'>
                    <td><b>Docker Version:</b></td><td>{sys_info.get('ServerVersion', 'Unknown')}</td>
                </tr>
                <tr style='border-bottom: 1px solid #cbd5e1; padding: 6px;'>
                    <td><b>Operating System:</b></td><td>{sys_info.get('OperatingSystem', 'Unknown')} ({sys_info.get('Architecture', '')})</td>
                </tr>
                <tr style='border-bottom: 1px solid #cbd5e1; padding: 6px;'>
                    <td><b>Total CPUs Allocated:</b></td><td>{sys_info.get('NCPU', 0)} / {multiprocessing.cpu_count()} Host Cores</td>
                </tr>
                <tr style='border-bottom: 1px solid #cbd5e1; padding: 6px;'>
                    <td><b>Total Memory Available:</b></td><td>{sys_info.get('MemTotal', 0) / (1024**3):.2f} GB</td>
                </tr>
                <tr style='border-bottom: 1px solid #cbd5e1; padding: 6px;'>
                    <td><b>Containers Count:</b></td><td>{sys_info.get('Containers', 0)} Total | 🟢 {sys_info.get('ContainersRunning', 0)} Active</td>
                </tr>
                <tr style='border-bottom: 1px solid #cbd5e1; padding: 6px;'>
                    <td><b>Active Volumes:</b></td><td>{len(vols)} mounts</td>
                </tr>
                <tr style='border-bottom: 1px solid #cbd5e1; padding: 6px;'>
                    <td><b>Active Networks:</b></td><td>{len(nets)} bridges</td>
                </tr>
            </table>
            """
            display(HTML(html_stats))
            log_to_console("System details queried successfully.", "success")
        except Exception as e:
            display(HTML(f"<div class='warning-card'>⚠️ Error querying system details: {e}</div>"))


In [ ]:
# ==============================================================================
# 5. LIVE MONITOR THREAD CONTROLLER
# ==============================================================================

def stream_live_metrics():
    global live_monitor_active
    while live_monitor_active:
        target = dropdown_containers.value
        with ui_monitor_panel:
            clear_output()
            if not target or target == "none":
                display(HTML("<div class='status-badge status-stopped'>⚠️ Select a target container to start metrics collection</div>"))
            else:
                try:
                    c = docker_client.containers.get(target)
                    c.reload()
                    
                    status = c.status.upper()
                    badge_class = "status-running" if status == "RUNNING" else "status-stopped"
                    if status == "PAUSED":
                        badge_class = "status-paused"
                        
                    # Stats query (timeout fast if stopped)
                    stats_data = "--"
                    if status == "RUNNING":
                        try:
                            raw_stats = c.stats(stream=False)
                            mem_use = raw_stats['memory_stats'].get('usage', 0) / (1024**2)
                            max_lim = raw_stats['memory_stats'].get('limit', 1) / (1024**2)
                            pct = (mem_use / max_lim) * 100 if max_lim > 0 else 0
                            stats_data = f"{mem_use:.2f} MB / {max_lim:.1f} MB ({pct:.2f}% utilization)"
                        except Exception:
                            stats_data = "Unavailable (retrying...)"
                    
                    html_panel = f"""
                    <div style='font-family: system-ui;'>
                        <div style='margin-bottom: 8px;'>🎯 Target Container: <b>{c.name}</b></div>
                        <div style='margin-bottom: 8px;'>💡 Status: <span class='status-badge {badge_class}'>{status}</span></div>
                        <div>🧠 Memory Resource Usage: <b>{stats_data}</b></div>
                    </div>
                    """
                    display(HTML(html_panel))
                except Exception as e:
                    display(HTML(f"<div style='color:#b91c1c;'>⚠️ Error streaming container updates: {e}</div>"))
        time.sleep(slider_refresh.value)

def handle_monitor_toggle(change):
    global live_monitor_active, live_monitor_thread
    if change['new']:
        live_monitor_active = True
        toggle_monitor.description = "⏹️ Stop Live Monitor"
        toggle_monitor.button_style = "danger"
        live_monitor_thread = threading.Thread(target=stream_live_metrics, daemon=True)
        live_monitor_thread.start()
        log_to_console("Active Background Monitor thread launched successfully.", "info")
    else:
        live_monitor_active = False
        toggle_monitor.description = "📊 Enable Live Monitor"
        toggle_monitor.button_style = "info"
        with ui_monitor_panel:
            clear_output()
            display(HTML("<div style='color: #64748b;'>Monitor offline. Press enable to start streaming metrics again.</div>"))
        log_to_console("Background Monitor thread terminated.", "info")

toggle_monitor.observe(handle_monitor_toggle, 'value')


In [7]:
# ==============================================================================
# 6. EVENT HOOKS & LAUNCH HANDLERS
# ==============================================================================

# Button Click Handlers Mapping
for btn in [btn_power_start, btn_power_stop, btn_power_pause, btn_power_unpause, btn_power_restart, btn_power_remove]:
    btn.on_click(handle_container_action)
    
btn_refresh_list.on_click(lambda b: [populate_containers(), log_to_console("Container list populated cleanly.", "success")])
btn_launch_container.on_click(deploy_new_container)
btn_refresh_logs.on_click(refresh_logs_panel)
btn_refresh_system.on_click(refresh_system_panel)

# UI Render Entry Point
if is_connected:
    populate_containers()
    refresh_logs_panel()
    refresh_system_panel()
    
    # Javascript Shortcut Fix to stop keydowns from executing cell shortcuts
    js_fix = widgets.HTML("""
    <script>
    document.querySelectorAll('textarea, input').forEach(el => {
        el.addEventListener('keydown', e => e.stopPropagation(), true);
    });
    </script>
    """)
    
    display(js_fix)
    display(main_dashboard)
    log_to_console("Mai AI Container Controller Initialized successfully. Ready to run.", "success")
else:
    btn_retry = widgets.Button(description="🔄 Retry Connection", button_style="warning")
    def retry_conn(b):
        global is_connected
        is_connected = connect_docker()
        clear_output()
        if is_connected:
            # Redraw full dashboard
            populate_containers()
            refresh_logs_panel()
            refresh_system_panel()
            display(widgets.HTML("""
            <script>
            document.querySelectorAll('textarea, input').forEach(el => {
                el.addEventListener('keydown', e => e.stopPropagation(), true);
            });
            </script>
            """))
            display(main_dashboard)
            log_to_console("Successfully re-established Docker Daemon connection!", "success")
        else:
            display(HTML("""
            <div class='warning-card'>
                <h3>⚠️ Docker Daemon Connection Unreachable</h3>
                <p>Could not initialize Docker client. Please verify Docker Desktop is running locally and active, then press the retry button below.</p>
            </div>
            """))
            display(btn_retry)
            
    btn_retry.on_click(retry_conn)
    
    display(HTML("""
    <div class='warning-card'>
        <h3>⚠️ Docker Daemon Connection Unreachable</h3>
        <p>Could not initialize Docker client. Please verify Docker Desktop is running locally and active, then press the retry button below.</p>
    </div>
    """))
    display(btn_retry)


HTML(value="\n    <script>\n    document.querySelectorAll('textarea, input').forEach(el => {\n        el.addEv…